In [1]:
%cd ../..

/home/eli/AnacondaProjects/epych


In [2]:
%env DASK_LOGGING__DISTRIBUTED=CRITICAL
%env OMPI_MCA_btl_sm_backing_directory=/mnt/data/tmp_storage
%env SPYTMPDIR=/mnt/data/tmp_storage
%env SPYLOGLEVEL=CRITICAL
%env SPYPARLOGLEVEL=CRITICAL

env: DASK_LOGGING__DISTRIBUTED=CRITICAL
env: OMPI_MCA_btl_sm_backing_directory=/mnt/data/tmp_storage
env: SPYTMPDIR=/mnt/data/tmp_storage
env: SPYLOGLEVEL=CRITICAL
env: SPYPARLOGLEVEL=CRITICAL


In [3]:
import collections
import functools
import logging
import numpy as np
import os
import quantities as pq

import epych
from epych.statistics import alignment

[striatum:1561715] shmem: mmap: an error occurred while determining whether or not /tmp/ompi.striatum.1000/jf.0/1300103168/shared_mem_cuda_pool.striatum could be created.
[striatum:1561715] create_and_attach: unable to create shared memory BTL coordinating structure :: size 134217728 


In [4]:
%matplotlib inline

In [5]:
logging.basicConfig(level=logging.INFO)

In [6]:
CONDITIONS = ["lonaive", "go_gloexp", "go_seqctl", "lo_gloexp", "lo_rndctl", "igo_seqctl"]
CONDITION_TITLES = {
    "lonaive": "Local Oddball Cue Trials",
    "go_gloexp": "Global Oddball",
    "go_seqctl": "AAAA Control",
    "lo_gloexp": "Local Oddball",
    "lo_rndctl": "AAAB Random",
    "igo_seqctl": "BBBB Control"
}

In [7]:
def samplings(cond):
    logging.info("Loading LFPs for condition %s" % cond)
    sampling = epych.recording.Sampling.unpickle("/mnt/data/000253/grandcat_%s" % cond)
    logging.info("Loaded LFPs for condition %s, baseline from (%f, %f)" % (cond, -0.250, -0.05))
    yield sampling.smap(lambda sig: sig.downsample(4))
    del sampling

In [8]:
def initialize_spectrum(key, signal, path=None):
    area = os.path.commonprefix([loc for loc in signal.channels.location])
    return epych.statistics.spectrum.Spectrogram(signal.df, signal.channels, signal.f0, taper="hann", path=path + "/" + key, time_window=0.200)

In [9]:
summaries = {}

In [10]:
for cond in CONDITIONS:
    logging.info("Calculating LFP spectrograms across condition %s" % cond)
    cond_path = "/mnt/data/000253/grand_spectrogram_downsample4_%s" % cond
    if os.path.exists(cond_path):
        continue
    summary = epych.statistic.Summary(alignment.location_prefix, functools.partial(initialize_spectrum, path=cond_path))
    summary.calculate(samplings(cond))
    summary.pickle(cond_path)
    summaries[cond] = summary

INFO:root:Calculating LFP spectrograms across condition lonaive
INFO:root:Calculating LFP spectrograms across condition go_gloexp
INFO:root:Calculating LFP spectrograms across condition go_seqctl
INFO:root:Calculating LFP spectrograms across condition lo_gloexp
INFO:root:Calculating LFP spectrograms across condition lo_rndctl
INFO:root:Calculating LFP spectrograms across condition igo_seqctl
